# Python and C++ extension

## Importing library

In [ ]:
import numpy as np
import argparse
import os.path
import scipy.sparse
import vtk
from pypolydim import polydim, gedim
from pypolydim.export_vtk_utilities import ExportVTKUtilities
from pypolydim.assembler_utilities import assembler_utilities

import sys
sys.path.insert(1, '../')
import other_utilities as other_ut

### Initialize

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument('-order','--method-order',dest='method_order', default=1, type=int, help="Method order")
parser.add_argument('-method','--method-type',dest='method_type', default=0, type=int, help="Method type")
parser.add_argument('-test', '--test-id', dest='test_id', default=1, type=int, help="Test type")
parser.add_argument('-mesh', '--mesh-type', dest='mesh_type', default=0, type=int, help="Mesh type")
parser.add_argument('-tol1', '--tolerance-1-d', dest='tolerance1_d', default=1.0e-6, type=float, help="Geometric Tolerance 1D")
parser.add_argument('-tol2', '--tolerance-2-d', dest='tolerance2_d', default=1.0e-12, type=float, help="Geometric Tolerance 2D")
parser.add_argument('-area', '--mesh-max-relative-area', dest='max_relative_area', default=0.1, type=float, help="Mesh max relative area")
parser.add_argument('-export', '--export-path', dest='export_path', default='./Export', type=str, help="Export Path")
parser.add_argument('-import', '--import-path', dest='import_path', default='./', type=str, help="Mesh Import Path")
args = parser.parse_args([])

geometry_utilities_config = gedim.GeometryUtilitiesConfig()
geometry_utilities_config.tolerance1_d = args.tolerance1_d
geometry_utilities_config.tolerance2_d = args.tolerance2_d
geometry_utilities = gedim.GeometryUtilities(geometry_utilities_config)
mesh_utilities = gedim.MeshUtilities()
vtk_utilities = ExportVTKUtilities()

# Export folder
export_file_path = args.export_path
if not os.path.exists(export_file_path):
    os.makedirs(export_file_path)

# Mesh file path
export_mesh_path = args.export_path + "/Mesh"
if not os.path.exists(export_mesh_path):
    os.makedirs(export_mesh_path)

# Solution file path
export_solution_path = args.export_path + "/Solution"
if not os.path.exists(export_solution_path):
    os.makedirs(export_solution_path)

## Elliptic Equation

Solving the following equation on square $\bar{\Omega} = [0, 1] \times [0, 1]$

$$
\begin{cases}
\nabla \cdot (a \nabla u) + b \cdot \nabla u + c u = f & \text{in } \Omega\\
a \nabla u \cdot n_1 = g_1 & \text{in } \Gamma_{top}\\
a \nabla u \cdot n_2 = g_2 & \text{in } \Gamma_{right}\\
u = 1.1 & \text{in } ∂Ω∖ (\Gamma_{top} ∪ \Gamma_{right})
\end{cases}
$$

where $u = 16 xy(1-x)(1-y) + 1.1$.

In [ ]:
def Poisson_A():
	return 10.0
def Poisson_B():
	return 0.1
def Poisson_C():
	return 2.0

def Poisson_a(numPoints, points):
	values = np.ones(numPoints) * Poisson_A()
	return values.ctypes.data

def Poisson_b(numPoints, points):
	values = np.ones((2, numPoints)) * Poisson_B()
	return values.ctypes.data

def Poisson_c(numPoints, points):
	values = np.ones(numPoints) * Poisson_C()
	return values.ctypes.data

def Poisson_f(numPoints, points):
	matPoints = gedim.make_nd_matrix(points, (3, numPoints), np.double)
	values = Poisson_A() * 32.0 * (matPoints[1,:] * (1.0 - matPoints[1,:]) + matPoints[0,:] * (1.0 - matPoints[0,:])) + \
	Poisson_B() * 16.0 * (1.0 - 2.0 * matPoints[0,:]) * matPoints[1,:] * (1.0 - matPoints[1,:]) + \
	Poisson_B() * 16.0 * (1.0 - 2.0 * matPoints[1,:]) * matPoints[0,:] * (1.0 - matPoints[0,:]) + \
	Poisson_C() * 16.0 * (matPoints[1,:] * (1.0 - matPoints[1,:]) * matPoints[0,:] * (1.0 - matPoints[0,:])) + Poisson_C() * 1.1
	return values.ctypes.data

def Poisson_exactSolution(numPoints, points):
	matPoints = gedim.make_nd_matrix(points, (3, numPoints), np.double)
	values = 16.0 * (matPoints[1,:] * (1.0 - matPoints[1,:]) * matPoints[0,:] * (1.0 - matPoints[0,:])) + 1.1
	return values.ctypes.data

def Poisson_exactDerivativeSolution(direction, numPoints, points):
	matPoints = gedim.make_nd_matrix(points, (3, numPoints), np.double)

	if direction == 0:
		values = 16.0 * (1.0 - 2.0 * matPoints[0,:]) * matPoints[1,:] * (1.0 - matPoints[1,:])
	elif direction == 1:
		values = 16.0 * (1.0 - 2.0 * matPoints[1,:]) * matPoints[0,:] * (1.0 - matPoints[0,:])
	else:
		values = np.zeros(numPoints)

	return values.ctypes.data

def Poisson_strongTerm(numPoints, points):
	matPoints = gedim.make_nd_matrix(points, (3, numPoints), np.double)
	values = 16.0 * (matPoints[1,:] * (1.0 - matPoints[1,:]) * matPoints[0,:] * (1.0 - matPoints[0,:])) + 1.1
	return values.ctypes.data

def Poisson_weakTerm_right(numPoints, points):
	matPoints = gedim.make_nd_matrix(points, (3, numPoints), np.double)
	values = Poisson_A() * 16.0 * (1.0 - 2.0 * matPoints[0,:]) * matPoints[1,:] * (1.0 - matPoints[1,:])
	return values.ctypes.data
	
def Poisson_weakTerm_left(numPoints, points):
	matPoints = gedim.make_nd_matrix(points, (3, numPoints), np.double)
	values = - Poisson_A() * 16.0 * (1.0 - 2.0 * matPoints[0,:]) * matPoints[1,:] * (1.0 - matPoints[1,:])
	return values.ctypes.data

### Define Simulation Parameters

We define the square domain and the simulation parameters

In [ ]:
pde_domain = polydim.pde_tools.mesh.pde_mesh_utilities.PDE_Domain_2D()
pde_domain.vertices = np.array([[0.0, 1.0, 1.0, 0.0],
                                [0.0, 0.0, 1.0, 1.0],
                                [0.0, 0.0, 0.0, 0.0]])
pde_domain.shape_type = polydim.pde_tools.mesh.pde_mesh_utilities.PDE_Domain_2D.Domain_Shape_Types.parallelogram
pde_domain.area = 1.0

In [ ]:
mesh_type = polydim.pde_tools.mesh.pde_mesh_utilities.MeshGenerator_Types_2D(args.mesh_type)
method_type = polydim.pde_tools.local_space_pcc_2_d.MethodTypes(args.method_type)
method_order = args.method_order
mesh_size = float(args.max_relative_area)

### Create Mesh

We create the mesh using the domain definition contained in the `pde_domain` object and the `mesh_size` selected.

In [ ]:
mesh_data = gedim.MeshMatrices()
mesh = gedim.MeshMatricesDAO(mesh_data)

polydim.pde_tools.mesh.pde_mesh_utilities.create_mesh_2_d(geometry_utilities,
                                                          mesh_utilities,
                                                          mesh_type,
                                                          pde_domain,
                                                          mesh_size,
                                                          mesh)
mesh_geometric_data = polydim.pde_tools.mesh.pde_mesh_utilities.compute_mesh_2_d_geometry_data(geometry_utilities, mesh_utilities, mesh)

vtk_utilities.export_mesh(export_mesh_path, mesh)

To description of the domain borders are passed to the library for the vertices and the edges of the domain as integer values called `markers`.
Each `marker` identifies a different boundary condition.

__In this example__:

 `marker=2` identifies $Γ_{right}$, `marker=4` identifies $Γ_{top}$ and `marker=1` identifies the Dirichlet boundary.

In [ ]:
info_internal = polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo(polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo.BoundaryTypes.none)
info_internal.marker = 0

info_dirichlet = polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo(polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo.BoundaryTypes.strong)
info_dirichlet.marker = 1

info_neumann_top = polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo(polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo.BoundaryTypes.weak)
info_neumann_top.marker = 4

info_neumann_right = polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo(polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo.BoundaryTypes.weak)
info_neumann_right.marker = 2

info_neumann_none = polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo(polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo.BoundaryTypes.none)
info_neumann_none.marker = 0

boundary_info = {
    0: info_internal,
    1: info_dirichlet,
    2: info_dirichlet,
    3: info_neumann_none,
    4: info_dirichlet,
    5: info_dirichlet,
    6: info_neumann_right,
    7: info_neumann_top,
    8: info_dirichlet
}

### Create Discrete Space FEM

The boundary condition types are passed to the library during the creation of the discrete space.
The types are the following:
* `BoundaryConditionType=1`: internal mesh point;
* `BoundaryConditionType=2`: strong boundary mesh point (Dirichlet in this example);
* `BoundaryConditionType=3`: weak boundary mesh point (Neumann in this example).
The array `BoundaryConditionType` describes for each `marker` the type of boundary condition associated.

__NB__: the array has dimension `num_markers+1`, as the first element is associated to non-usable `marker=0`.

__In this example__:

we have $3$ different markers, thus `BoundaryConditionsType` has size $3+1=4$. In particular `marker=1` has type `BoundaryConditionsType[1]=2` (Dirichlet), `marker=2` has type `BoundaryConditionsType[2]=3` and `marker=3` (Neumann) has type `BoundaryConditionsType[3]=3` (Neumann).

In [ ]:
reference_element_data = polydim.pde_tools.local_space_pcc_2_d.create_reference_element(method_type, method_order)

mesh_connectivity_data = polydim.pde_tools.mesh.MeshMatricesDAO_mesh_connectivity_data(mesh)

dof_manager = polydim.pde_tools.do_fs.DOFsManager()
mesh_dofs_info = polydim.pde_tools.local_space_pcc_2_d.set_mesh_do_fs_info(reference_element_data, mesh, boundary_info)
dofs_data = dof_manager.create_do_fs_2_d(mesh_dofs_info, mesh_connectivity_data)

### Assemble linear system

In [ ]:
def exact_solution_function(x, y, z):
    polynomial = x + y + 0.5
    result = 1.0
    for i in range(method_order):
        result *= polynomial
    return result

u_exact = polydim.pde_tools.assembler_utilities.pcc_2_d.assembler_exact_solution(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       dofs_data,
                                                                       reference_element_data,
                                                                       exact_solution_function)
print("u_ex:", u_exact.exact_solution)
print("u_ex_D:", u_exact.exact_solution_strong)
                                                                       

In [ ]:
def source_term_function(x, y, z, u):  
    polynomial = x + y + 0.5
    result = 2.0 * method_order * (method_order - 1)
    max_order = method_order - 2;
    for i in range(max_order):
        result *= polynomial
    return -result

source_term = polydim.pde_tools.assembler_utilities.pcc_2_d.assembler_source_term(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       dofs_data,
                                                                       reference_element_data,
                                                                       source_term_function)
print("f:", source_term)
                                                                       

In [ ]:
def strong_solution_function(marker, x, y, z):  
    return exact_solution_function(x, y, z)

strong_solution = polydim.pde_tools.assembler_utilities.pcc_2_d.assembler_strong_solution(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       mesh_dofs_info,
                                                                       dofs_data,
                                                                       reference_element_data,
                                                                       strong_solution_function)
print("u_D:", strong_solution)
                                                                       

In [ ]:
def diffusion_term_function(x, y, z, u):  
    return 1.0

elliptic_operator = polydim.pde_tools.assembler_utilities.pcc_2_d.assembler_elliptic_operator(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       dofs_data,
                                                                       reference_element_data,
                                                                       diffusion_term_function)


A = other_ut.make_np_sparse(elliptic_operator.a)
A_D = other_ut.make_np_sparse(elliptic_operator.a_strong)

print("A:", A)
print("A_D:", A_D)

In [ ]:
rhs = source_term - A_D @ strong_solution
u = scipy.sparse.linalg.spsolve(A, rhs)
print("u:", u)

In [ ]:
post_process_data = polydim.pde_tools.assembler_utilities.pcc_2_d.assembler_post_process(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       dofs_data,
                                                                       reference_element_data,
                                                                                         u,
                                                                                         strong_solution,
                                                                       exact_solution_function)

print("error_L2:", post_process_data.error_l2)
print("numeric_norm_L2:", post_process_data.numeric_norm_l2)
print("exact_norm_L2:", post_process_data.exact_norm_l2)
print("cell0Ds_size:", post_process_data.cell0_ds_numeric.shape)
print("mesh_vertices: ", mesh.cell0_d_total_number())
print("cell0Ds_size:", post_process_data.cell0_ds_exact.shape)
print("mesh_vertices: ", mesh.cell0_d_total_number())

In [ ]:
vtk_utilities.export_solution_2(export_solution_path + '/Solution.vtu',
                                mesh, 
                                post_process_data.cell0_ds_numeric,
                                post_process_data.cell0_ds_exact,
                                post_process_data.cell2_ds_error_l2,
                               post_process_data.cell2_ds_error_l2)

In [ ]:
[stiffness, stiffnessStrong] = gedim.AssembleStiffnessMatrix(Poisson_a, problemData, lib)

[advection, advectionStrong] = gedim.AssembleAdvectionMatrix(Poisson_b, problemData, lib)

[reaction, reactionStrong] = gedim.AssembleReactionMatrix(Poisson_c, problemData, lib)

forcingTerm = gedim.AssembleForcingTerm(Poisson_f, problemData, lib)

solutionStrong = gedim.AssembleStrongSolution(Poisson_strongTerm, 1, problemData, lib)

weakTerm_right = gedim.AssembleWeakTerm(Poisson_weakTerm_right, 2, problemData, lib)
weakTerm_left = gedim.AssembleWeakTerm(Poisson_weakTerm_left, 3, problemData, lib)

### Solve linear system

In [ ]:
solution = gedim.LUSolver(stiffness + advection + reaction, \
    forcingTerm - \
    (stiffnessStrong + advectionStrong + reactionStrong) @ solutionStrong + \
    weakTerm_right + \
    weakTerm_left, lib)

In [ ]:
gedim.PlotSolution(mesh, dofs, strongs, solution, solutionStrong)

In [ ]:
gedim.ExportSolution(Poisson_exactSolution, solution, solutionStrong, lib)

### Compute errors

In [ ]:
errorL2 = gedim.ComputeErrorL2(Poisson_exactSolution, solution, solutionStrong, lib)

errorH1 = gedim.ComputeErrorH1(Poisson_exactDerivativeSolution, solution, solutionStrong, lib)

print("dofs", "h", "errorL2", "errorH1")
print(problemData['NumberDOFs'], '{:.16e}'.format(problemData['H']), '{:.16e}'.format(errorL2), '{:.16e}'.format(errorH1))

## Heat Conductivity Equation

Solving the following equation on square $\bar{\Omega} = [-1, +1] \times [-1, +1]$

$$
\begin{cases}
\nabla \cdot (k \nabla u) = 0 & \text{in } \Omega\\
k \nabla u \cdot n_1 = g & \text{in } \Gamma_{down}\\
u = 0 & \text{in } \Gamma_{up}\\
k \nabla u \cdot n_2 = 0 & \text{otherwise} 
\end{cases}
$$

where $k = k_1$ if $x^2 + y^2 \leq R^2$ and $k = 1$ otherwise. 

In [ ]:
def Heat_R():
	return 0.5
def Heat_K():
	return 6.68
def Heat_G():
	return 0.94

def Heat_k(numPoints, points):
	matPoints = gedim.make_nd_matrix(points, (3, numPoints), np.double)
	values = np.ones(numPoints)
	for p in range(0, numPoints):
		if (matPoints[0,p] * matPoints[0,p] + matPoints[1,p] * matPoints[1,p]) <= (Heat_R() * Heat_R() + 1.0e-16):
			values[p] = Heat_K()
	return values.ctypes.data

def Heat_weakTerm_down(numPoints, points):
	values = np.ones(numPoints) * Heat_G()
	return values.ctypes.data

def Empty_exactSolution(numPoints, points):
	values = np.zeros(numPoints)
	return values.ctypes.data

### Define Simulation Parameters

In [ ]:
order = 2

### Import Mesh

In [ ]:
%%writefile ImportMesh.csv
InputFolderPath
../../CppToPython/Meshes/Mesh2

In [ ]:
[meshInfo, mesh] = gedim.ImportDomainMesh2D(lib)

In [ ]:
gedim.PlotMesh(mesh)

### Create Discrete Space FEM

In [ ]:
discreteSpace = { 'Order': order, 'Type': 1, 'BoundaryConditionsType': [1, 3, 3, 2] }
[problemData, dofs, strongs] = gedim.Discretize(discreteSpace, lib)

In [ ]:
gedim.PlotDofs(mesh, dofs, strongs)

### Assemble linear system

In [ ]:
[stiffness, stiffnessStrong] = gedim.AssembleStiffnessMatrix(Heat_k, problemData, lib)
	
weakTerm_down = gedim.AssembleWeakTerm(Heat_weakTerm_down, 1, problemData, lib)

### Solve linear system

In [ ]:
solution = gedim.LUSolver(stiffness, weakTerm_down, lib)

In [ ]:
gedim.PlotSolution(mesh, dofs, strongs, solution, np.zeros(problemData['NumberStrongs']))

In [ ]:
gedim.ExportSolution(Empty_exactSolution, solution, np.zeros(problemData['NumberStrongs']), lib)